# TP2 : descente de gradient, choix du pas et convergence
**Cours :** Algorithmes d'optimisation, séance 6 · **Intervenant :** Serigne Faye

## Objectifs
- implémenter la descente de gradient avec différentes règles de choix du pas ;
- étudier l'influence du pas fixe sur la convergence ;
- implémenter une recherche linéaire (backtracking) et le pas optimal ;
- comparer les critères d'arrêt et plusieurs configurations ;
- appliquer la descente de gradient à un problème d'apprentissage (régression linéaire).

## Consignes
Exécutez les cellules dans l'ordre, complétez les cellules marquées **À COMPLÉTER** et répondez aux questions dans les cellules de texte.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

## Partie 1 : la descente de gradient
On cherche à minimiser $f$ en itérant $x_{k+1} = x_k - t_k \nabla f(x_k)$. La fonction `gradient_descent` ci-dessous accepte un pas fixe (un nombre) ou une **règle de pas** (une fonction `regle(x, g)` renvoyant le pas $t_k$). Elle propose trois critères d'arrêt :
- `"grad"` : $\|\nabla f(x_k)\| < \text{tol}$ ;
- `"pas"` : $\|x_{k+1} - x_k\| < \text{tol}$ ;
- `"fval"` : $|f(x_{k+1}) - f(x_k)| < \text{tol}$.

**Exercice 1.1.** Compléter les deux lignes marquées.

In [ ]:
def gradient_descent(f, grad, x0, step, max_iter=10000, tol=1e-6, crit="grad"):
    x = np.array(x0, dtype=float)
    xs, fs, gn, ts = [x.copy()], [f(x)], [np.linalg.norm(grad(x))], []
    converged = False
    for k in range(max_iter):
        g = grad(x)
        if crit == "grad" and np.linalg.norm(g) < tol:
            converged = True; break
        t = None                         # À COMPLÉTER : step(x, g) si step est une fonction, sinon step
        x_new = None                     # À COMPLÉTER : mise à jour x - t * g
        if not np.all(np.isfinite(x_new)) or np.linalg.norm(x_new) > 1e8:
            break                        # divergence
        small_step = np.linalg.norm(x_new - x) < tol
        small_f = abs(f(x_new) - f(x)) < tol
        x = x_new
        xs.append(x.copy()); fs.append(f(x)); gn.append(np.linalg.norm(grad(x))); ts.append(t)
        if (crit == "pas" and small_step) or (crit == "fval" and small_f):
            converged = True; break
    return x, {"xs": np.array(xs), "fs": np.array(fs), "gn": np.array(gn), "ts": np.array(ts),
               "iters": len(xs) - 1, "converged": converged}

On teste sur $f(x, y) = x^2 + 10y^2$, de gradient $(2x, 20y)^\top$, en partant de $(1, 1)$.

In [ ]:
def f(p): return p[0]**2 + 10*p[1]**2
def grad(p): return np.array([2*p[0], 20*p[1]])
x0 = np.array([1.0, 1.0])

x, h = gradient_descent(f, grad, x0, step=0.05)
print(x, h["iters"], h["converged"])   # attendu : proche de (0, 0)

## Partie 2 : influence d'un pas fixe
**Exercice 2.1.** Pour chaque pas de la liste, lancer la descente (`max_iter=2000`) et afficher le nombre d'itérations, si l'algorithme a convergé et la valeur finale de $f$. Tracer ensuite $f(x_k)$ en échelle logarithmique pour chaque pas.

In [ ]:
pas_liste = [0.001, 0.01, 0.05, 0.09, 0.1, 0.11]
resultats = {}
for t in pas_liste:
    # À COMPLÉTER : lancer gradient_descent et ranger l'historique dans resultats[t]
    pass

print("   pas | itérations | convergé |   f finale")
# À COMPLÉTER : afficher une ligne par pas

plt.figure(figsize=(8, 5))
# À COMPLÉTER : plt.semilogy(resultats[t]["fs"], label=f"pas = {t}") pour chaque pas
plt.xlabel("itération"); plt.ylabel("f(x_k)"); plt.legend(); plt.grid(True); plt.show()

**Question 2.2.** Classez les pas : lequel est le plus rapide ? lequel est trop lent ? lequel diverge ? Que se passe-t-il pour $t = 0{,}1$ ? Comparez avec la condition $t < 2/L$ où $L = 20$ est la plus grande courbure de $f$ (valeur propre maximale de la hessienne $\mathrm{diag}(2, 20)$).

**Réponse :** *(à compléter)*

## Partie 3 : recherche linéaire (backtracking)
Plutôt que de fixer $t$, on le choisit à chaque itération : on part de $t = 1$ et on le divise par 2 tant que la condition d'Armijo n'est pas satisfaite :
$$f(x - t g) \leq f(x) - c\, t\, \|g\|^2, \qquad g = \nabla f(x), \quad c = 10^{-4}.$$

**Exercice 3.1.** Écrire `backtracking(f, grad, x, t0, beta, c)` puis lancer la descente avec la règle `lambda x, g: backtracking(f, grad, x)`.

In [ ]:
def backtracking(f, grad, x, t0=1.0, beta=0.5, c=1e-4):
    # À COMPLÉTER : boucle while qui divise t par 2 tant que la condition d'Armijo n'est pas vérifiée
    raise NotImplementedError("À compléter")

x_bt, h_bt = gradient_descent(f, grad, x0, step=lambda x, g: backtracking(f, grad, x))
print(h_bt["iters"], h_bt["converged"], x_bt)

**Question 3.2.** Comparez le nombre d'itérations avec le meilleur pas fixe de la partie 2. Quel est l'avantage du backtracking ? Quel est son coût ?

**Réponse :** *(à compléter)*

## Partie 4 : pas optimal pour une fonction quadratique
Pour $f(x) = \tfrac12 x^\top A x$ avec $A = \mathrm{diag}(2, 20)$ (c'est notre fonction $x^2 + 10y^2$), la recherche linéaire exacte dans la direction $-g$ donne le pas
$$t_k = \frac{g^\top g}{g^\top A g}.$$

**Exercice 4.1.** Implémenter cette règle, lancer la descente depuis le point $(1\,;\,0{,}1)$ (choisi pour bien voir le comportement) et tracer sur les courbes de niveau les trajectoires obtenues avec le pas optimal et avec $t = 0{,}05$.

In [ ]:
A = np.diag([2.0, 20.0])
regle_optimale = None    # À COMPLÉTER : lambda x, g: ...
x0b = np.array([1.0, 0.1])

x_opt, h_opt = gradient_descent(f, grad, x0b, step=regle_optimale)
x_fix, h_fix = gradient_descent(f, grad, x0b, step=0.05)
print("pas optimal :", h_opt["iters"], "itérations ; pas 0.05 :", h_fix["iters"], "itérations")

xs = np.linspace(-1.2, 1.2, 200); X, Y = np.meshgrid(xs, xs)
plt.figure(figsize=(6, 6)); plt.contour(X, Y, X**2 + 10*Y**2, levels=20, alpha=0.4)
# À COMPLÉTER : tracer les itérés (h_opt["xs"] et h_fix["xs"]) avec plt.plot(..., "o-", ...)
plt.legend(); plt.show()

**Question 4.2.** Décrivez les trajectoires. Pourquoi « zigzaguent-elles » dans une vallée étroite ?

**Réponse :** *(à compléter)*

## Partie 5 : influence du conditionnement
On considère $f_\kappa(x, y) = \tfrac12 (x^2 + \kappa y^2)$, de hessienne $\mathrm{diag}(1, \kappa)$. Le **conditionnement** est $\kappa$.

**Exercice 5.1.** Pour $\kappa \in \{1, 10, 100\}$, en partant de $(1\,;\,1/\kappa)$, comparer le nombre d'itérations (critère `"grad"`, `tol=1e-6`) avec : un pas fixe $2/(1+\kappa)$, le backtracking et le pas optimal.

In [ ]:
def quadratique(kappa):
    A = np.diag([1.0, kappa])
    return (lambda p: 0.5 * p @ A @ p), (lambda p: A @ p), A

print(" kappa | pas fixe | backtracking | pas optimal")
for kappa in [1, 10, 100]:
    fk, gk, Ak = quadratique(kappa)
    xk = np.array([1.0, 1.0 / kappa])
    # À COMPLÉTER : trois descentes depuis xk et affichage des nombres d'itérations

**Question 5.2.** Comment le nombre d'itérations évolue-t-il avec $\kappa$ ? Pourquoi un mauvais conditionnement est-il un problème ?

**Réponse :** *(à compléter)*

## Partie 6 : une fonction difficile (Rosenbrock)
$f(x, y) = (1 - x)^2 + 100 (y - x^2)^2$, de minimum $(1, 1)$, à minimiser depuis $(-1{,}2\, ;\, 1)$.

**Exercice 6.1.** Comparer : pas fixe $10^{-3}$ (20 000 itérations maximum), backtracking (50 000 itérations maximum, `tol=1e-6`) et `scipy.optimize.minimize` avec la méthode `"BFGS"` (méthode quasi-Newton, vue à la séance suivante).

In [ ]:
rosen = lambda p: (1 - p[0])**2 + 100*(p[1] - p[0]**2)**2
rgrad = lambda p: np.array([-2*(1 - p[0]) - 400*p[0]*(p[1] - p[0]**2), 200*(p[1] - p[0]**2)])
r0 = np.array([-1.2, 1.0])

# À COMPLÉTER : trois résolutions, puis affichage du point final et du nombre d'itérations de chacune

**Question 6.2.** Classez les trois méthodes. Que laisse penser le résultat de BFGS sur l'intérêt d'utiliser de l'information du second ordre ?

**Réponse :** *(à compléter)*

## Partie 7 : application à l'apprentissage (régression linéaire)
On dispose de $N = 50$ points $(x_i, y_i)$ et on cherche la droite $y = a x + b$ qui minimise l'erreur quadratique moyenne
$$L(a, b) = \frac{1}{N} \sum_{i=1}^{N} (a x_i + b - y_i)^2 .$$

**Exercice 7.1.** Écrire `L(p)` et `grad_L(p)` pour $p = (a, b)$, avec $\nabla L = \left(\dfrac{2}{N}\sum (a x_i + b - y_i) x_i,\; \dfrac{2}{N}\sum (a x_i + b - y_i)\right)^\top$. Lancer la descente de gradient avec un pas fixe $0{,}1$ depuis $(0, 0)$ et comparer avec `np.polyfit`.

In [ ]:
rng = np.random.default_rng(0)
xd = np.linspace(0, 1, 50)
yd = 2*xd + 1 + 0.1*rng.normal(size=50)

def L(p):
    # À COMPLÉTER
    raise NotImplementedError("À compléter")
def grad_L(p):
    # À COMPLÉTER
    raise NotImplementedError("À compléter")

# À COMPLÉTER : descente de gradient (pas 0.1, max_iter=5000, tol=1e-8), affichage de (a, b) et de np.polyfit(xd, yd, 1)
# Puis tracer les données et la droite obtenue, ainsi que la courbe de la perte

**Exercice 7.2.** Refaire la descente avec les pas $0{,}01$, $0{,}5$ et $0{,}9$ (`max_iter=3000`). Pour chacun, indiquer si l'algorithme converge et combien d'itérations sont utilisées.

In [ ]:
for t in [0.01, 0.5, 0.9]:
    # À COMPLÉTER
    pass

**Question 7.3.** Quel rôle joue le pas, appelé ici *learning rate* en apprentissage automatique ? Que se passe-t-il s'il est trop petit, trop grand ?

**Réponse :** *(à compléter)*

## Partie 8 : synthèse
**Question 8.1.** Pour chacun des critères d'arrêt `"grad"`, `"pas"` et `"fval"`, donnez un avantage et une limite. Lequel choisiriez-vous en pratique, et avec quelle sécurité supplémentaire ?

**Réponse :** *(à compléter)*

**Question 8.2.** Résumez en 5 lignes comment choisir entre pas fixe, backtracking et pas optimal.

**Réponse :** *(à compléter)*